---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

# 🏎️ LazyPredict Benchmarking
**Extended · Data Pattern Recognition for the Rest of Us**

> Before committing to a model, benchmark dozens of algorithms in one call. LazyPredict tells you which model families are worth deeper investment — in minutes instead of days.

### Dataset
**Telecom Customer Churn**
Predict which customers will cancel their service. Churn costs telecom companies billions annually — retaining customers is 5-25x cheaper than acquiring new ones. This is a classic imbalanced binary classification problem with real business stakes.

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

!pip install lazypredict -q
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Telecom Churn Dataset
np.random.seed(42)
n = 7043
tenure = np.random.randint(0, 72, n)
monthly = np.random.uniform(18, 120, n)
contract = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
senior = np.random.binomial(1, 0.16, n)
log_odds = -1.5 - 0.05*tenure + 0.015*monthly + 1.2*(contract=='Month-to-month').astype(float) + 0.2*senior
churn = (np.random.uniform(0,1,n) < 1/(1+np.exp(-log_odds))).astype(int)
telco = pd.DataFrame({'tenure':tenure,'MonthlyCharges':monthly.round(2),
                       'Contract':contract,'SeniorCitizen':senior,'Churn':churn})
print(f'✓ Telecom Churn: {telco.shape}  Churn rate: {telco["Churn"].mean():.1%}')

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — What LazyPredict Does

In [ ]:
# Telecom Churn Dataset (IBM Telco — widely used in industry)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
print(f"\nTelecom Churn dataset: {len(y):,} customers")
print(f"Churn rate: {y.mean():.1%}")
print(f"Training: {len(y_tr):,}  |  Test: {len(y_te):,}")
print("\nRunning LazyPredict on all sklearn classifiers...")
print("(This benchmarks 30+ models — may take 2-3 minutes)")

## 🔬 Part 2 — Running the Benchmark

In [ ]:
# Run LazyPredict — benchmark all classifiers
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_tr, X_te, y_tr, y_te)

print("=== LazyPredict Results — Telecom Churn (sorted by AUC-ROC) ===\n")
# Show top 15
display_cols = [c for c in ['Accuracy','Balanced Accuracy','ROC AUC','F1 Score','Time Taken']
                if c in models_df.columns]
if 'ROC AUC' in models_df.columns:
    models_sorted = models_df.sort_values('ROC AUC', ascending=False)
else:
    models_sorted = models_df.sort_values('Accuracy', ascending=False)
print(models_sorted.head(15)[display_cols].to_string())

## 📊 Part 3 — Interpreting the Leaderboard

In [ ]:
# Visualize the benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# AUC-ROC comparison
metric = 'ROC AUC' if 'ROC AUC' in models_df.columns else 'Accuracy'
top_models = models_sorted.head(15)
colors_bar = ['#1a7a45' if i < 3 else '#1e5fa8' if i < 8 else '#888'
              for i in range(len(top_models))]
axes[0].barh(range(len(top_models)), top_models[metric].values[::-1],
             color=colors_bar[::-1], edgecolor='white')
axes[0].set_yticks(range(len(top_models)))
axes[0].set_yticklabels([n[:25] for n in top_models.index[::-1]], fontsize=9)
axes[0].set_xlabel(metric)
axes[0].set_title(f'LazyPredict: {metric} by Model\n'
                  f'Green = top 3, Blue = top 8')
axes[0].axvline(top_models[metric].median(), color='#e85d20', lw=1.5, ls='--',
               label=f'Median = {top_models[metric].median():.3f}')
axes[0].legend()

# Speed vs accuracy tradeoff
if 'Time Taken' in models_df.columns:
    axes[1].scatter(models_df['Time Taken'], models_df[metric],
                    color='#1e5fa8', alpha=0.7, s=40)
    for i, (name, row) in enumerate(models_df.iterrows()):
        if row[metric] > models_df[metric].quantile(0.85):
            axes[1].annotate(name[:15], (row['Time Taken'], row[metric]),
                            fontsize=7, alpha=0.8)
    axes[1].set_xlabel('Training Time (seconds)')
    axes[1].set_ylabel(metric)
    axes[1].set_title('Speed vs Performance Tradeoff\n'
                      'Best = top-left (fast AND accurate)')

plt.tight_layout(); plt.show()
best_model = models_sorted.index[0]
print(f"\n\u2714 LazyPredict winner: {best_model}")
print(f"   This is a STARTING POINT — now tune this model with proper cross-validation")
print(f"   LazyPredict does NOT replace rigorous model evaluation!")

## ✅ Part 4 — Selecting Models for Tuning

In [ ]:
# Business framing: what does AUC mean for churn?
print("=== Business Interpretation ===\n")
print("A churn model with AUC=0.85 means:")
print("  If you randomly pick 1 churner and 1 non-churner,")
print("  the model ranks the churner higher 85% of the time.")
print()
print("At a retention budget of $50/customer and 7,043 customers:")
churn_rate = y.mean()
n_churners = int(len(y) * churn_rate)
budget_per_customer = 50
print(f"  Total churners to prevent: ~{n_churners}")
print(f"  If model catches 70% of churners at a 2:1 false positive rate:")
caught = int(n_churners * 0.70)
contacted = caught * 3  # 2:1 FP ratio
cost = contacted * budget_per_customer
prevented_revenue = caught * 120 * 12  # $120/mo avg, 1 year
print(f"  Customers contacted: {contacted:,}")
print(f"  Campaign cost: ${cost:,}")
print(f"  Revenue retained (est.): ${prevented_revenue:,}")
print(f"  Net ROI: {(prevented_revenue - cost)/cost*100:.0f}%")

## 💼 Exercise

Run LazyPredict on the Telecom Churn dataset. Which model achieves the highest F1 score? Is it the same as the highest accuracy model? Select the top 3 models and describe what makes each suitable for this problem.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — LazyPredict Benchmarking
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is LazyPredict's primary use case and what is it NOT good for?
# @markdown **Q2:** Why is accuracy a poor metric for churn prediction (hint: class imbalance)?
# @markdown **Q3:** The top LazyPredict model uses default hyperparameters. What should you do next?
# @markdown **Q4:** How would you explain AUC-ROC to a non-technical marketing manager?
# @markdown **Q5:** For churn: would you rather minimize false positives or false negatives? Why?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="LazyPredict Benchmarking"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*